In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(context='notebook', style='darkgrid')

In [2]:
def gaussian_surface(x_in, centers, A, covs):
    """
    Returns the height at point `x_in` for a surface defined with peaks/valleys of height A at locations defined by `center`. 
    
    x_in:    (N, 1, 2, 1) - The point where you want to measure height.
    centers: (1, M, 2, 1) - Defines the location of the peak or peaks of the surface. 
    A:       (1, M)       - Defines the height of the peaks
    covs:    (1, M, 2, 2) - Defines the spread or shape of the peaks. 
    """
    
    d = x_in - centers
    cov_inv = np.linalg.inv(covs)

    q = d.transpose(0, 1, 3, 2) @ cov_inv @ d
    q = q.squeeze((-1, -2))

    heights = A * np.exp(-0.5 * q)

    return heights.sum(axis=1)


def normal_pdf(x, mean=0.0, std=1.0):
    """ 1D Normal Distribution """
    variance = std ** 2
    denominator = np.sqrt(2 * np.pi * variance)
    numerator = np.exp(-((x - mean) ** 2) / (2 * variance))
    return numerator / denominator


def multivariate_normal_pdf(x: np.ndarray, mu: np.ndarray, cov):
    """
    multivariate normal pdf.

    Computes the broadcasted multivariate probability density of all x with respect to all mu, therefore
    x and mu must have shapes compatable for broadcasting. 
    
    x: input with shape (ndim, ...)
    mu: mean with shape (ndim, ...)
    cov: Covariance with shape (ndim, ndim)
    """

    dim = x.shape[0]

    cov_inv = np.linalg.inv(cov)
    cov_det = np.linalg.det(cov)
    
    d = x - mu                 # (ndim, ...)
    d = np.moveaxis(d, 0, -1)  # (..., ndim)


    coefficient = (2 * np.pi)**(-dim / 2) * cov_det**(-1/2)
    
    quadratic = (
        d[..., None, :]   # (..., 1, ndim)
        @ cov_inv         # (ndim, ndim)    -- out -->  (..., 1, ndim)
        @ d[..., :, None] # (..., ndim, 1)  -- out -->  (..., 1, 1)
    ).squeeze((-1, -2))      

    return coefficient * np.exp(-0.5 * quadratic)

### Define bathymetry
Define an example surface to traverse over. 

In [31]:
W, H = (150, 150)

x = np.linspace(-1, 1, W)
y = np.linspace(-1, 1, H)

mesh = np.meshgrid(x, y)
xv, yv = mesh

# (N, 1, 2, 1)
coords = np.stack([xv.ravel(), yv.ravel()], axis=1)
coords = coords[:, None, :, None]

# Centers: (1, M, 2, 1)
centers = np.array([
    [[-0.3], [ 0.0]],
    [[ 0.5], [ 0.3]],
    [[ 0.0], [-0.6]],
])[None]

# Heights: (1, M)
# Positive = peak, negative = valley
A = np.array([[10.0, 7.0, -4.0]])

# One covariance matrix per peak/valley: (1, M, 2, 2)
covs = np.array([
    [[0.08, 0.00],
     [0.00, 0.04]],

    [[0.2, 0.00],
     [0.00, 0.1]],

    [[0.15, 0.00],
     [0.00, 0.05]],
])[None]

z = gaussian_surface(coords, centers, A, covs).reshape(H, W)

### Define Probablistic and Dynamics Models

In [4]:
# fixed timestep and dynamics for constant velocity
dt = 0.1
v = np.asarray([0.25, 0.25])

# Process and Measurement Covariance
Q = np.asarray([[1e-4, 0], [0, 1e-4]])
R = 0.5

# Dynamics Model
def F(x, v, dt):
    return x + dt * v

### Calculate Transition Tensor
This is re-used at every timestep to perform the prediction step. It is the most costly part of this algorithm. 

In [5]:
# Create a grid grid containing all possible permutations of state components... in this case positions (x_i, y_j)  
# For i in [0, ... W]) and j in [0 ... H]
x = np.linspace(-1, 1, W)
y = np.linspace(-1, 1, H)

# state[k, i, j] gives the value of state component k (k=0 for x, k=1 for y) for state permutation (x_i, y_j)
state_space = np.stack(np.meshgrid(x, y), axis=0)

# shape (2, H, W)
# transitions[k, i, j] is the result of applying dynamics component k of the state permutation (x_i, x_j)
transitions = F(state_space, v[:, None,None], dt)

state_space = state_space[:, None, None, ...] # (2, y_new, x_new) ---> (2, 1, 1, y_new, x_new)
transitions = transitions[..., None, None]    # (2, y_old, x_old) ---> (2, y_old, x_old, 1, 1)

# p(x_k | x_(k-1))
# transition_distribution[i, j, :, :] is the conditional density distribution given old position (x_j, y_i).
# transition_distribution[i, j, k, l] is the probability density of transitioning from original point (x_j, y_i) to new point (x_l, y_k).
transition_tensor = multivariate_normal_pdf(state_space, transitions, Q)

### Define Likelihood, Update and Prediction Functions

In [6]:
likelihood = lambda obs: normal_pdf(obs, z, R)

def update_belief(measurement, prior):
    update = likelihood(measurement) * prior
    update /= update.sum()
    return update

def predict_next(prior):
    prediction = transition_tensor * prior[:, :, None, None]
    prediction = prediction.sum((0, 1))
    prediction /= prediction.sum()
    return prediction

### Run the Algorithm Loop

In [53]:
from scipy.interpolate import RegularGridInterpolator

interpolate_depth = RegularGridInterpolator((x, y), z)

# Start with Uniform prior
prior = np.zeros_like(z) + 1 / (H * W)

# Start at bottom left of map
x_k = np.array([-1, -1])

posteriors = [prior]
true_position = [x_k]
for i in np.arange(100):
    
    # Simulate measurement
    x_k = F(x_k, v, dt)
    if (x_k >= 1).any():
        break
        
    measurement = interpolate_depth(x_k)

    # Update belief
    prior = update_belief(measurement, prior)

    # book keeping for plots
    true_position.append(x_k)
    posteriors.append(prior)

    # Predict next state
    prior = predict_next(prior)

In [54]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(8, 6))

# Set heatmap axis labels
xtick_ixs = np.concat([np.arange(0, W, 20), [W - 1]])
xtick_labels = [f"{v:.2f}" for v in x[xtick_ixs]]

ytick_ixs = np.concat([np.arange(0, H, 20), [H - 1]])
ytick_labels = [f"{v:.2f}" for v in y[ytick_ixs]]

def update(frame_idx):
    ax.clear()

    sns.heatmap(posteriors[frame_idx], ax=ax, cmap="viridis", cbar=False)

    # plot true position
    xx, yy = true_position[frame_idx]

    ix = np.argmin(np.abs(x - xx))
    iy = np.argmin(np.abs(y - yy))

    sns.scatterplot(x=[ix + 0.5], y=[iy + 0.5], s=100, color="red", marker='x', ax=ax)

    ax.set_xticks(xtick_ixs, labels=xtick_labels, rotation=45)
    ax.set_yticks(ytick_ixs, labels=ytick_labels, rotation=0)
    ax.set_title(f"Time: {frame_idx * dt:2f}")
    ax.invert_yaxis()

anim = FuncAnimation(
    fig,
    update,
    frames=len(posteriors),
    interval=200,
    repeat=True,
)


plt.close(fig)
HTML(anim.to_jshtml())